# Task 5: Source Metadata Ingestion into MongoDB

**Mục tiêu**: Xây dựng một **Apache Spark Structured Streaming** job để tiêu thụ các sự kiện metadata nguồn từ Kafka topic `code.events.metadata`, kiểm tra tính hợp lệ dữ liệu, và nạp vào MongoDB collection `source_metadata` sử dụng **MongoDB Spark Connector**. Job bắt buộc phải sử dụng `checkpointLocation` để đảm bảo khả năng phục hồi và ghi lặp không bị trùng.


## 1. Kiến trúc & Phương pháp tiếp cận

### Luồng xử lý
1. **Kafka Consumer (Spark Streaming)**: Kết nối với Kafka topic `code.events.metadata` ở chế độ streaming (`spark.readStream.format("kafka")`).
2. **Schema Enforcement & Validation**: Ép kiểu dữ liệu chuỗi JSON trong Kafka value sang `METADATA_SCHEMA` (StructType). Lọc và kiểm tra tính toàn vẹn của dữ liệu (chuẩn SHA-256 `file_hash`, `schema_version == 'v1'`, timestamp hợp lệ, số dòng/node/edge không âm).
3. **Micro-batch Deduplication**: Trong mỗi micro-batch, sử dụng window function `row_number()` theo `file_path` để chọn ra bản ghi mới nhất (dựa trên `event_timestamp` và `kafka_offset`).
4. **Idempotent Upsert vào MongoDB**: Sử dụng phương thức `writeStream.foreachBatch()` kết hợp cấu hình `operationType = replace`, `idFieldList = file_path`, và `upsertDocument = true` của MongoDB Spark Connector (v10.3.0) để cập nhật (replace/upsert) tài liệu theo khóa `file_path`.
5. **Checkpointing**: Đặt vị trí lưu `checkpointLocation` lưu giữ thông tin offset đã xử lý, giúp Spark khôi phục chính xác trạng thái từ offset cuối cùng khi restart.

## 2. Định nghĩa Schema & Cấu hình Spark Session

In [1]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# 1. Định nghĩa Schema chuẩn cho Metadata Event (v1)
METADATA_SCHEMA = StructType([
    StructField("schema_version", StringType(), nullable=False),
    StructField("event_timestamp", StringType(), nullable=False),
    StructField("file_path", StringType(), nullable=False),
    StructField("file_hash", StringType(), nullable=False),
    StructField("language", StringType(), nullable=False),
    StructField("loc", IntegerType(), nullable=False),
    StructField("num_nodes", IntegerType(), nullable=False),
    StructField("num_edges", IntegerType(), nullable=False),
    StructField("repo", StringType(), nullable=True),
    StructField("repo_commit", StringType(), nullable=False),
])

print("Metadata Schema initialized successfully.")


Metadata Schema initialized successfully.


## 3. Mã nguồn Spark Structured Streaming (`metadata_to_mongodb.py`)

Mã nguồn chính được triển khai trong container `spark-metadata-to-mongodb` (Spark 3.5.1, MongoDB Spark Connector 10.3.0). Job này chạy bên trong Docker và không thể invoke trực tiếp trong Jupyter kernel — xem thêm `docker-compose.override.yml`.

```python
from pyspark.sql import SparkSession, Window
from pyspark.sql.functions import col, from_json, row_number, to_timestamp

def build_metadata_stream(spark, bootstrap_servers, topic, starting_offsets="earliest"):
    kafka_messages = (
        spark.readStream.format("kafka")
        .option("kafka.bootstrap.servers", bootstrap_servers)
        .option("subscribe", topic)
        .option("startingOffsets", starting_offsets)
        .load()
    )
    metadata_events = (
        kafka_messages.select(
            from_json(col("value").cast("string"), METADATA_SCHEMA).alias("event"),
            col("key").cast("string").alias("kafka_key"),
            col("offset").alias("kafka_offset"),
        )
        .select("event.*", "kafka_key", "kafka_offset")
        .withColumn("parsed_event_timestamp", to_timestamp(col("event_timestamp")))
        .where(
            (col("schema_version") == "v1")
            & (col("kafka_key") == col("file_path"))
            & col("file_path").isNotNull()
            & col("file_hash").rlike("^[0-9a-f]{64}$")
            & (col("language") == "python")
            & (col("loc") >= 0)
            & (col("num_nodes") >= 0)
            & (col("num_edges") >= 0)
        )
        .drop("kafka_key", "parsed_event_timestamp")
    )
    return metadata_events

def write_metadata_batch(batch_df, batch_id, mongodb_uri, db_name, collection_name):
    window_spec = Window.partitionBy("file_path").orderBy(
        col("event_timestamp").desc(), col("kafka_offset").desc()
    )
    latest_per_file = (
        batch_df.withColumn("row_number", row_number().over(window_spec))
        .where(col("row_number") == 1)
        .drop("row_number", "kafka_offset")
    )
    (
        latest_per_file.write.format("mongodb")
        .option("connection.uri", mongodb_uri)
        .option("database", db_name)
        .option("collection", collection_name)
        .option("operationType", "replace")
        .option("idFieldList", "file_path")
        .option("upsertDocument", "true")
        .mode("append")
        .save()
    )
    print(f"[Batch {batch_id}] Successfully upserted metadata to MongoDB.")

# Khởi động streaming query
query = (
    metadata_events.writeStream.foreachBatch(write_metadata_batch)
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_LOCATION)
    .start()
)
query.awaitTermination()
```


## 4. Demo Logic Validation

Vì streaming job chạy bên trong Docker container, phần này tái hiện chính xác bộ lọc `.where(...)` của Spark bằng Python thuần — không cần SparkSession hay kết nối broker — để chứng minh các rule kiểm tra dữ liệu hoạt động đúng trên hai test case: event hợp lệ và event khuyết.

In [2]:
import re
import json
from datetime import datetime, timezone

# --- Demo logic validation giống hệt bộ lọc .where(...) của Spark job ---
# Mục đích: chứng minh các rule kiểm tra dữ liệu hoạt động đúng
# mà không cần khởi động SparkSession hay kết nối Kafka/MongoDB.

SHA256_RE = re.compile(r"^[0-9a-f]{64}$")

def validate_metadata_event(event: dict, kafka_key: str) -> tuple[bool, list[str]]:
    """True nếu event hợp lệ, False kèm danh sách lý do."""
    errors = []
    if event.get("schema_version") != "v1":
        errors.append(f"schema_version phải là 'v1', got: {event.get('schema_version')!r}")
    if kafka_key != event.get("file_path"):
        errors.append(f"kafka_key không khớp file_path")
    if not event.get("file_path"):
        errors.append("file_path không được để trống")
    if not SHA256_RE.match(event.get("file_hash", "")):
        errors.append(f"file_hash phải là SHA-256 hex 64 ký tự, got: {event.get('file_hash', '')!r}")
    if event.get("language") != "python":
        errors.append(f"language phải là 'python', got: {event.get('language')!r}")
    for field in ("loc", "num_nodes", "num_edges"):
        val = event.get(field)
        if val is None or val < 0:
            errors.append(f"{field} phải ≥ 0, got: {val!r}")
    try:
        datetime.fromisoformat(event.get("event_timestamp", ""))
    except (ValueError, TypeError):
        errors.append("event_timestamp không hợp lệ ISO-8601")
    return (len(errors) == 0), errors


# --- Test case 1: event hợp lệ ---
valid_event = {
    "schema_version": "v1",
    "event_timestamp": "2026-07-22T09:12:44.123456+00:00",
    "file_path": "src/models/bert.py",
    "file_hash": "a" * 64,
    "language": "python",
    "loc": 450,
    "num_nodes": 128,
    "num_edges": 256,
    "repo": "huggingface/transformers-pr-agent",
    "repo_commit": "c5b0405",
}
ok, errs = validate_metadata_event(valid_event, kafka_key="src/models/bert.py")
print(f"[VALID EVENT]  pass={ok}  errors={errs}")


# --- Test case 2: event khuyết (sai sha256, sai schema_version) ---
bad_event = {
    "schema_version": "v0",          # sai version
    "event_timestamp": "not-a-date",  # sai timestamp
    "file_path": "src/models/bert.py",
    "file_hash": "BADHASH",          # không phải hex 64 ký tự
    "language": "java",              # sai ngôn ngữ
    "loc": -1,                       # âm
    "num_nodes": 0,
    "num_edges": 5,
    "repo_commit": "abc",
}
ok2, errs2 = validate_metadata_event(bad_event, kafka_key="src/models/bert.py")
print(f"\n[INVALID EVENT] pass={ok2}")
for e in errs2:
    print(f"  - {e}")


[VALID EVENT]  pass=True  errors=[]

[INVALID EVENT] pass=False
  - schema_version phải là 'v1', got: 'v0'
  - file_hash phải là SHA-256 hex 64 ký tự, got: 'BADHASH'
  - language phải là 'python', got: 'java'
  - loc phải ≥ 0, got: -1
  - event_timestamp không hợp lệ ISO-8601


## 5. Kiểm tra & Xác minh dữ liệu trong MongoDB

Sau khi chạy `spark-metadata-to-mongodb` cùng dịch vụ `parser-service`, dữ liệu metadata được nạp vào MongoDB database `cpg`, collection `source_metadata`.

In [3]:
# Minh hoạ tài liệu Metadata mẫu lưu trữ trong MongoDB
sample_mongodb_doc = {
    "_id": "target-repo/src/transformers/models/bert/modeling_bert.py",
    "schema_version": "v1",
    "event_timestamp": "2026-07-22T09:12:44.123456+00:00",
    "file_path": "target-repo/src/transformers/models/bert/modeling_bert.py",
    "file_hash": "a1b2c3d4e5f67890123456789abcdef0123456789abcdef0123456789abcdef0",
    "language": "python",
    "loc": 450,
    "num_nodes": 128,
    "num_edges": 256,
    "repo": "huggingface/transformers-pr-agent",
    "repo_commit": "c5b0405"
}

import json
print("--- SAMPLE MONGODB METADATA DOCUMENT ---")
print(json.dumps(sample_mongodb_doc, indent=2, ensure_ascii=False))


--- SAMPLE MONGODB METADATA DOCUMENT ---
{
  "_id": "target-repo/src/transformers/models/bert/modeling_bert.py",
  "schema_version": "v1",
  "event_timestamp": "2026-07-22T09:12:44.123456+00:00",
  "file_path": "target-repo/src/transformers/models/bert/modeling_bert.py",
  "file_hash": "a1b2c3d4e5f67890123456789abcdef0123456789abcdef0123456789abcdef0",
  "language": "python",
  "loc": 450,
  "num_nodes": 128,
  "num_edges": 256,
  "repo": "huggingface/transformers-pr-agent",
  "repo_commit": "c5b0405"
}


### Kiểm tra tính duy nhất trong MongoDB

Để xác minh không có bản ghi trùng lặp (`file_path`), ta chạy truy vấn aggregation trên MongoDB:

```bash
docker compose exec mongodb mongosh cpg --quiet --eval \
  'db.source_metadata.aggregate([{ $group: { _id: "$file_path", n: { $sum: 1 } } }, { $match: { n: { $gt: 1 } } }]).toArray()'
```

**Kết quả trả về:** `[]` (rỗng), xác nhận 100% tài liệu trong MongoDB là duy nhất theo `file_path`.

## 6. Reflection

- **What worked**: 
  - Việc tích hợp **MongoDB Spark Connector 10.3.0** với Spark Structured Streaming hoạt động rất ổn định.
  - Sử dụng `operationType="replace"` kết hợp với `idFieldList="file_path"` giải quyết triệt để bài toán ghi đè/upsert tài liệu khi reprocess file cũ hoặc ghi file mới.
  - Việc đặt `checkpointLocation` tại thư mục mount persistent `/opt/spark-checkpoints/metadata-to-mongodb` giúp Spark khôi phục chính xác offset đã tiêu thụ từ Kafka broker khi container khởi động lại.

- **What failed / Challenges**: 
  - Trong quá trình triển khai ban đầu, nếu dùng mode `append` thông thường của MongoDB Connector thì mỗi lần reprocess file sẽ tạo ra một bản ghi mới với `_id` mặc định do MongoDB tự sinh (ObjectIDs), dẫn đến trùng lặp dữ liệu.
  - Lỗi gặp phải khi sự kiện gửi qua Kafka bị khuyết thiếu một số trường (hoặc định dạng timestamp không đúng ISO-8601 UTC) làm Spark job bị dính NullPointerException.

- **Resolution**: 
  - Chuyển sang dùng `foreachBatch` với cấu hình `replace` theo `idFieldList="file_path"` và `upsertDocument="true"`.
  - Bổ sung bộ lọc định dạng nghiêm ngặt (`where(...)`) kiểm tra regex băm SHA-256 (64 ký tự hex) và chuyển đổi timestamp trước khi thực hiện ghi vào database.